In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2 as cv
from PIL import Image

import torch
import torch.nn as nn
from torch.nn import functional as F

import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms import v2

In [ ]:
print(f"Qurilma: {'GPU' if torch.cuda.is_available() else 'CPU'}")
print(f"Qurilma nomi: {torch.cuda.get_device_name(0)}")

Qurilma: GPU
Qurilma nomi: Tesla T4


In [ ]:
# Pretrained ResNet50 yuklash

weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights).to('cuda')

for param in model.parameters():
  param.requires_grad = False

for name, param in model.named_parameters():
    print(
        name,
        "Shape:", param.shape,
        "Device:", param.device,
        "Trainable:", param.requires_grad
    )

conv1.weight Shape: torch.Size([64, 3, 7, 7]) Device: cuda:0 Trainable: False
bn1.weight Shape: torch.Size([64]) Device: cuda:0 Trainable: False
bn1.bias Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.conv1.weight Shape: torch.Size([64, 64, 1, 1]) Device: cuda:0 Trainable: False
layer1.0.bn1.weight Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.bn1.bias Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.conv2.weight Shape: torch.Size([64, 64, 3, 3]) Device: cuda:0 Trainable: False
layer1.0.bn2.weight Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.bn2.bias Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.conv3.weight Shape: torch.Size([256, 64, 1, 1]) Device: cuda:0 Trainable: False
layer1.0.bn3.weight Shape: torch.Size([256]) Device: cuda:0 Trainable: False
layer1.0.bn3.bias Shape: torch.Size([256]) Device: cuda:0 Trainable: False
layer1.0.downsample.0.weight Shape: torch.Size([256, 64, 1, 1]) Device: c

In [ ]:
# Qayta train uchun modeling FC parametrlani trainga ochamiz

model.fc = nn.Linear(in_features = model.fc.in_features, out_features = 10).to('cuda')

for name, fc in model.fc.named_parameters():
  print(f"Name: {name}, Shape: {fc.shape}, Device: {fc.device}, Trainable: {fc.requires_grad}")

Name: weight, Shape: torch.Size([10, 2048]), Device: cuda:0, Trainable: True
Name: bias, Shape: torch.Size([10]), Device: cuda:0, Trainable: True


In [ ]:
# preprocess = weights.transforms() # Tayyoridan foydalanish uchun

preprocess =  v2.Compose([
    v2.Resize(size=232, interpolation=v2.InterpolationMode.BILINEAR, antialias=True),
    v2.CenterCrop(size=224),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(preprocess)

Compose(
      Resize(size=[232], interpolation=InterpolationMode.BILINEAR, antialias=True)
      CenterCrop(size=(224, 224))
      ToImage()
      ToDtype(scale=True)
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)


In [ ]:
model.fc

Linear(in_features=2048, out_features=1000, bias=True)

In [ ]:
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

In [ ]:
class CNN(nn.Module):
  def __init__(self):
    super().__init__()
    self.ReLU = nn.ReLU()
    self.conv1 = nn.Conv2d(3,32,5,1)
    self.MaxPoll = nn.MaxPool2d(3,2)
    self.conv2 = nn.Conv2d(32,64,7,2)
    self.conv3 = nn.Conv2d(64,128,2,2)
    self.fc1 = nn.LazyLinear(1024)
    self.fc2 = nn.Linear(1024, 10)
    self.Flatten = nn.Flatten()

  def forward(self,x):

    x = self.conv1(x)
    x = self.MaxPoll(x)
    x = self.ReLU(x)

    x = self.conv2(x)
    x = self.MaxPoll(x)
    x = self.ReLU(x)

    x = self.conv3(x)
    x = self.MaxPoll(x)
    x = self.ReLU(x)

    x = self.Flatten(x)

    x = self.fc1(x)
    x = self.ReLU(x)

    x = self.fc2(x)

    return (x)

model = CNN()
print(model)

class CNN2(nn.Module):
  def __init__(self):
    super().__init__()

    self.net = nn.Sequential(
        nn.Conv2d(3,32,7,1),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Conv2d(32,128,5,1),
        nn.MaxPool2d(2,2),
        nn.ReLU(),

        nn.Conv2d(128,64,3,2),
        nn.MaxPool2d(3,3),
        nn.ReLU(),

        nn.Flatten(),
        nn.LazyLinear(1024),
        nn.ReLU(),
        nn.Linear(1024,10))
  def forward(self, x):
    return self.net(x)

model2 = CNN2()
print(model2)

CNN(
  (ReLU): ReLU()
  (conv1): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1))
  (MaxPoll): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(7, 7), stride=(2, 2))
  (conv3): Conv2d(64, 128, kernel_size=(2, 2), stride=(2, 2))
  (fc1): LazyLinear(in_features=0, out_features=1024, bias=True)
  (fc2): Linear(in_features=1024, out_features=10, bias=True)
  (Flatten): Flatten(start_dim=1, end_dim=-1)
)
CNN2(
  (net): Sequential(
    (0): Conv2d(3, 32, kernel_size=(7, 7), stride=(1, 1))
    (1): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (2): ReLU()
    (3): Conv2d(32, 128, kernel_size=(5, 5), stride=(1, 1))
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): ReLU()
    (6): Conv2d(128, 64, kernel_size=(3, 3), stride=(2, 2))
    (7): MaxPool2d(kernel_size=3, stride=3, padding=0, dilation=1, ceil_mode=False)
    (8): ReLU()
    (9): Flatten(start_dim=